# Day 14 · Borgesian map — แผนที่ที่ละเอียดจนกลายเป็นดินแดน

**#30DayMapChallenge 2026 · 14 พ.ย. 2569** · โจทย์ทางการ: *"A map so detailed it becomes the territory."*
*(หัวข้อ "Borgesian map" อ้างถึง *On Exactitude in Science* ของ Jorge Luis Borges พิมพ์ครั้งแรกปี 1946)*

โน้ตบุ๊กนี้ **วัด** คำพังเพยของ Borges เป็นตัวเลขสองเส้น ไม่ใช่วาดแผนที่สวย ๆ หนึ่งแผ่น

| เส้น | วัดอะไร | ผล |
|---|---|---|
| ราคาของแผนที่ | จำนวนเซลล์ · จำนวนจุดที่ต้องวาด · ขนาดไฟล์ · เวลาวาด | โตขึ้น ~7 เท่าทุกขั้น ไม่อิ่มตัว |
| ข้อมูลที่ได้เพิ่ม | สัดส่วนเซลล์ที่มีจุดทะเบียน · คำตอบ "จุดที่ใกล้ที่สุด" ที่เปลี่ยนไปเมื่อซอย | อิ่มตัวตั้งแต่ res 5–6 |

**ต้องรันตามลำดับ cell** — สคริปต์ตรวจตัวเองก่อนวาดทุกครั้ง (guard ของซีรีส์อยู่ใน `tools/guards.py`)
และเขียนผลออกมาเป็น `day14_ladder.csv` + `day14_stats.json` ซึ่งเป็นแหล่งตัวเลขเดียวที่โพสต์อ้างถึง

**หมายเหตุความซื่อสัตย์ 3 ข้อ**

1. `h3` ไม่ใช่ package ที่ติดมากับ Colab มาตรฐาน — cell แรกจะ `pip install h3==4.5.0` ให้
2. ห้องผู้ตัดสิน `guards.py` ตอนนี้อยู่ใน repo ที่ `tools/guards.py` แล้ว — บนเครื่องตัวเองสคริปต์ใช้ไฟล์นั้น
   บน Colab จะดาวน์โหลดจาก GitHub มาใช้ ถ้าดึงไม่ได้สคริปต์จะล้มเหลวเสียงดัง ไม่ได้ข้าม guard เงียบ ๆ
3. ตัวเลขที่ res 9 ขึ้นไป (รวมถึง "ผมหนึ่งเส้น") เป็น **เลขคณิตต่อจากกฎ 7 เท่า** ไม่ใช่แผนที่ที่วาดจริง
   ทุกจุดที่ต่อเอาจะถูกป้ายไว้ว่า extrapolate ทั้งในภาพและใน `day14_stats.json`

### ค่าคงที่ + ชุดเครื่องมือ (อ่าน/ดาวน์โหลดข้อมูล, ตั้งฟอนต์ไทย, ตรวจว่ารันที่ไหน)

In [ ]:
# -*- coding: utf-8 -*-
"""
Day 14 · Borgesian map — "A map so detailed it becomes the territory."
(ข้อความหัวข้อจาก 30daymapchallenge.com อ่านสดวันเขียน · อ้างถึงเรื่องสั้น
 "On Exactitude in Science" ของ Jorge Luis Borges พิมพ์ครั้งแรกปี 1946)

คำถามของแผ่นนี้ไม่ใช่ "ทำแผนที่ให้ละเอียดที่สุด" แต่คือวางสองเส้นเทียบกัน
  เส้นที่ 1  ราคาของแผนที่ (จำนวนเซลล์ · จำนวนจุดที่ต้องวาด · ขนาดไฟล์ · เวลาวาด)
  เส้นที่ 2  ข้อมูลที่แผนที่บอก (สัดส่วนเซลล์ที่มีจุดทะเบียน · คำตอบที่เปลี่ยนไปเมื่อซอยเพิ่ม
             · ความคลาดเคลื่อนของพื้นที่ประเทศเมื่อเทียบกับค่าอ้างอิง)
ราคาโตขั้นละเจ็ดเท่าไม่อิ่มตัว ส่วนข้อมูลอิ่มตัวตั้งแต่ res 5–6 และที่สำคัญกว่านั้น
ต้นทางข้อมูล (เส้นขอบประเทศใน Natural Earth 10m) มีเพดานของมันเอง: ตอนที่ตาราง
ละเอียดกว่าช่วงห่างของจุดบนเส้นขอบ รายละเอียดที่เพิ่มคือของแต่ง ไม่ใช่ของวัด

ทุกตัวเลขที่โพสต์อ้างอ่านจาก day14_ladder.csv / day14_stats.json ที่สคริปต์นี้เขียนออกเท่านั้น
"""
import os, sys, json, time, subprocess, urllib.request, zipfile
import numpy as np, pandas as pd, geopandas as gpd

T0 = time.time()
# รันได้ทั้งบนเครื่องผู้เขียนและบน Colab: ถ้าไม่มีโฟลเดอร์รีโปให้ ก็ดึงข้อมูลจาก GitHub / Natural Earth ตรง ๆ
COLAB = not os.path.isdir(os.path.join(r"D:\GIS30Days", "geo-notes"))
WORK = os.getcwd() if COLAB else r"D:\GIS30Days"
REPO = os.path.join(WORK, "geo-notes")
CACHE = os.path.join(WORK, "day14_cache") if COLAB else os.path.join(WORK, "cache", "day06")
OUT = os.path.join(WORK, "out", "day14")
os.makedirs(CACHE, exist_ok=True); os.makedirs(OUT, exist_ok=True)
RAW = "https://raw.githubusercontent.com/thammarat-ai/geo-notes/main/"

if COLAB:                                   # h3 ไม่ใช่ package ติดตั้งมาใน Colab มาตรฐาน
    try:
        import h3
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "h3==4.5.0"])

def ensure_shp(stem, sub="cultural"):
    shp = os.path.join(CACHE, stem + ".shp")
    if not os.path.exists(shp):
        zp = os.path.join(CACHE, stem + ".zip")
        urllib.request.urlretrieve(f"https://naciscdn.org/naturalearth/10m/{sub}/{stem}.zip", zp)
        zipfile.ZipFile(zp).extractall(CACHE)
    return shp

def repo_file(rel):
    dst = os.path.join(CACHE, os.path.basename(rel))
    if COLAB and not os.path.exists(dst):
        urllib.request.urlretrieve(RAW + rel, dst)
    return dst if COLAB else os.path.join(REPO, rel)

if COLAB:
    ensure_shp("ne_10m_admin_0_countries")
    for rel in ["ofl/trirong/Trirong-Regular.ttf", "ofl/trirong/Trirong-Bold.ttf"]:
        dst = os.path.join(CACHE, os.path.basename(rel))
        if not os.path.exists(dst):
            urllib.request.urlretrieve("https://github.com/google/fonts/raw/main/" + rel, dst)

import h3
from shapely.geometry import Point
from shapely.validation import make_valid
from pyproj import CRS, Transformer
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.collections import PolyCollection

try:
    sys.path.insert(0, os.path.join(WORK, "tools"))
    from guards import (guard_bbox, guard_crs, guard_area, guard_variance,
                        guard_distances, lint_thai, hav_km)
except ModuleNotFoundError:
    # บน Colab ไม่มีโฟลเดอร์ tools/ ของเวิร์กสเปซ — ดึงห้องผู้ตัดสินจาก repo มาใช้
    gp = os.path.join(CACHE, "guards.py")
    if not os.path.exists(gp):
        urllib.request.urlretrieve(RAW + "tools/guards.py", gp)
    sys.path.insert(0, CACHE)
    from guards import (guard_bbox, guard_crs, guard_area, guard_variance,
                        guard_distances, lint_thai, hav_km)

for _f in sorted(os.listdir(CACHE)):
    if _f.endswith(".ttf"):
        font_manager.fontManager.addfont(os.path.join(CACHE, _f))
plt.rcParams["font.family"] = "Trirong"
_LOGF = open(os.path.join(OUT, "day14_run.log"), "w", encoding="utf-8")
def log(m):
    s = f"[{time.time()-T0:6.1f}s] {m}"
    print(s, flush=True)
    _LOGF.write(s + chr(10)); _LOGF.flush()

EQ = CRS.from_proj4("+proj=eqdc +lat_0=14 +lon_0=101 +lat_1=8 +lat_2=20 +units=m +no_defs +ellps=WGS84")
BG, INK = "#0d1522", "#e8e3d3"
ACCENT, WARM, COOL, TEAL = "#f5b942", "#f4566a", "#38bdf8", "#2dd4bf"
MUTED = "#9fb2c8"

RES_MEASURE = [2, 3, 4, 5, 6, 7, 8]   # ระดับที่ H3 เติมให้จริงในเครื่องนี้
RES_DRAW = [2, 3, 4, 5, 6, 8]         # ระดับที่นำไปเขียนเป็นภาพ
BYTE_RES = [3, 4, 5, 6]               # ระดับที่เขียน GeoJSON จริงเพื่อวัดขนาดไฟล์
TIME_RES = [5, 6, 7, 8]               # ระดับที่จับเวลาวาดจริง
HAIR_M = 5e-5                         # ผมหนึ่งเส้น 0.05 มม.
DOT300_M = 25.4e-3 / 300.0            # จุดหมึกหนึ่งจุดที่ 300 dpi = 0.0847 มม.
A4_M2 = 0.210 * 0.297                 # พื้นที่แผ่น A4 ตามนิยาม ISO 216
HEX_K = 3 * np.sqrt(3) / 2            # พื้นที่หกเหลี่ยมด้านเท่า = 2.598 × s²

TXT = []                              # รวมข้อความไทยที่ไปโผล่ในภาพ ให้ lint จับ
def T(s):
    TXT.append(s)
    return s

## ตอน 1) ข้อมูลตั้งต้น

In [ ]:
# ---------------------------------------------------------------- 1. ข้อมูลตั้งต้น
ne = gpd.read_file(ensure_shp("ne_10m_admin_0_countries"))
row = ne[ne["NAME"] == "Thailand"]
assert len(row) == 1, f"ต้องเจอ Thailand 1 แถว ได้ {len(row)}"
TH = make_valid(row.geometry.iloc[0])
PARTS = list(TH.geoms) if TH.geom_type.startswith("Multi") else [TH]
country_m = gpd.GeoSeries([TH], crs="EPSG:4326").to_crs(EQ).area.iloc[0] / 1e6
REF_KM2 = 512593.1                    # ตัวเลขที่ซีรีส์ใช้มาตั้งแต่ Day 8 (Natural Earth 10m)
guard_area(country_m, REF_KM2, "thailand-area-recheck", tol=0.005)
guard_crs(TH.bounds, "thailand-outline-wgs84", expect="degrees")

df = pd.read_csv(repo_file("data/darksky_thailand.csv"))
df["name"] = df["name"].astype(str).str.strip()
SITES = np.column_stack([df.longitude.values, df.latitude.values])   # (lon, lat)
NAMES = df["name"].values
guard_bbox(SITES, 96.5, 107.6, 4.3, 21.7, "darksky-sites")
assert len(df) == 83, f"ทะเบียนต้องมี 83 จุด ได้ {len(df)}"
log(f"ประเทศ {country_m:,.1f} ตร.กม. ยืนยันซ้ำแล้ว · ทะเบียน {len(df)} จุด · outline {len(PARTS)} ส่วน")

## ตอน 2) แปลงขอบประเทศให้เป็นรูปที่ H3 รับ

In [ ]:
# ---------------------------------------------------------------- 2. แปลงขอบประเทศให้เป็นรูปที่ H3 รับ
def _ring(coords):
    out = []
    for c in coords:
        ll = (round(float(c[1]), 7), round(float(c[0]), 7))   # H3 รับ (lat, lng)
        if not out or out[-1] != ll:
            out.append(ll)
    if len(out) > 1 and out[0] == out[-1]:
        out.pop()
    return out

HPS = []
for p in PARTS:
    loops = [_ring(p.exterior.coords)] + [_ring(i.coords) for i in p.interiors]
    loops = [l for l in loops if len(l) >= 3]
    if loops:
        HPS.append(h3.LatLngPoly(loops[0], *loops[1:]))
MP = h3.LatLngMultiPoly(*HPS)

def fill(res):
    return np.array(sorted(h3.polygon_to_cells(MP, res)), dtype=object)

CELLS = {}
for r in RES_MEASURE:
    t1 = time.perf_counter()
    CELLS[r] = fill(r)
    log(f"เติม res {r}: {len(CELLS[r]):,} เซลล์ ({time.perf_counter()-t1:.2f} วิ)")

# กฎเจ็ดเท่า: H3 แบ่งแม่หนึ่งเซลล์เป็นลูกเจ็ดเซลล์ จำนวนเซลล์ต้องโตประมาณเจ็ดเท่าต่อขั้น
RATIO = {r: float(len(CELLS[r + 1]) / len(CELLS[r])) for r in RES_MEASURE[:-1]}
for r, v in RATIO.items():
    assert 6.0 <= v <= 7.3, f"res {r}→{r+1} อัตราส่วน {v:.2f} ไม่ใกล้ 7 = เติมไม่ครบหรือตกขอบ"
log("กฎเจ็ดเท่าผ่านการตรวจ: " + " · ".join(f"{r}→{r+1}={v:.2f}" for r, v in RATIO.items()))

## ตอน 3) ปรับยอดกับ Day 3 (1,738 เซลล์)

In [ ]:
# ------------------------------------------------ 3. ปรับยอดกับ Day 3 (1,738 เซลล์)
g3 = gpd.read_file(repo_file("posts/day03-polygons/day03_hex_polygons.geojson"))
day3 = set(g3["cell"].astype(str))
mine5 = set(CELLS[5].astype(str))
REC = dict(day3_cells=len(day3), res5_ne10m_cells=len(mine5), common=len(day3 & mine5),
           only_day3=len(day3 - mine5), only_today=len(mine5 - day3),
           reason="Day 3 เติมตารางจากขอบประเทศ GADM · วันนี้ใช้ Natural Earth 10m "
                  "(มาตรฐานที่ซีรีส์เริ่มใช้ตั้งแต่ Day 8) วิธีเดียวกัน คนละต้นทางของเส้นขอบ")
log(f"Day 3 {REC['day3_cells']} เซลล์ · วันนี้ {REC['res5_ne10m_cells']} เซลล์ · "
    f"ซ้ำกัน {REC['common']} · ต่าง {REC['only_day3']}/{REC['only_today']}")
assert REC["common"] >= 0.98 * min(REC["day3_cells"], REC["res5_ne10m_cells"]), \
    "เซลล์ res 5 ของสองวันควรซ้อนกันเกือบทั้งหมด ถ้าต่างมากกว่านี้แปลว่าวิธีเติมเปลี่ยน"

## ตอน 4) ตัวเลขต่อขั้นของบันได

In [ ]:
# ---------------------------------------------------------------- 4. ตัวเลขต่อขั้นของบันได
def areas_sample(cells, k=20000, seed=14):
    idx = np.random.default_rng(seed).choice(len(cells), size=min(k, len(cells)), replace=False)
    return float(np.mean([h3.cell_area(cells[i], "km^2") for i in idx])), int(len(idx))

def nearest_for(cells_arr, res):
    """ตอบ 'จุดทะเบียนที่ใกล้ที่สุด' ของทุกเซลล์ด้วยระยะทรงกลมจริง คำนวณเป็นก้อนกัน index เพี้ยน"""
    ll = np.array([h3.cell_to_latlng(x) for x in cells_arr])
    deg = np.column_stack([ll[:, 1], ll[:, 0]])
    guard_bbox(deg, 96.5, 107.6, 4.3, 21.7, f"cell-centre-res{res}")
    ans = np.empty(len(deg), dtype=object)
    dmin = np.empty(len(deg))
    step = 200000
    for i in range(0, len(deg), step):
        d = hav_km(deg[i:i + step], SITES)
        ans[i:i + step] = NAMES[d.argmin(axis=1)]
        dmin[i:i + step] = d.min(axis=1)
    guard_distances(dmin.reshape(-1, 1), f"cell-to-site-res{res}", max_km=600)
    return deg, ans, dmin

rows, ANS, CENT, DMIN = [], {}, {}, {}
OUTSIDE = {}
for r in RES_MEASURE:
    c = CELLS[r]
    n = len(c)
    a_mean, a_n = areas_sample(c)
    nominal = a_mean * n
    site_cell = [h3.latlng_to_cell(float(la), float(lo), r)
                 for la, lo in zip(df.latitude.values, df.longitude.values)]
    # นับเฉพาะเซลล์ที่อยู่ในชุดที่เติมจากประเทศ: จุดทะเบียนริมขอบอาจตกในเซลล์ยักษ์
    # ที่ center ของมันอยู่นอกรอยต่อ (เห็นชัดที่ res 2 ซึ่ง 13 เซลล์ที่มีจุด > 6 เซลล์ของประเทศ)
    setc = set(c.astype(str))
    inside = set(site_cell) & setc
    outside_names = sorted({str(NAMES[k]) for k, sc in enumerate(site_cell) if sc not in setc})
    OUTSIDE[r] = outside_names
    occ = len(inside)
    assert occ <= n, f"res {r}: เซลล์ที่มีจุด {occ} > จำนวนเซลล์ {n} = นับผิด"
    deg, ans, dmin = nearest_for(c, r)
    ANS[r], CENT[r], DMIN[r] = ans, deg, dmin
    rows.append(dict(res=r, cells=n, edge_km=float(h3.average_hexagon_edge_length(r, "km")),
                     cell_area_km2=a_mean, area_sampled=a_n, nominal_area_km2=nominal,
                     nominal_vs_country_pct=100 * nominal / REF_KM2,
                     occupied_cells=occ, occupied_pct=100 * occ / n,
                     site_cells_distinct=len(set(site_cell)),
                     site_cells_outside_fill=len(set(site_cell) - setc),
                     sites_without_own_cell=len(outside_names), vertices=n * 6))
    log(f"res {r}: {n:,} เซลล์ · {a_mean:,.4f} ตร.กม./เซลล์ · รวม {nominal:,.0f} "
        f"({100*nominal/REF_KM2:.2f}% ของประเทศ) · เซลล์ที่มีจุดทะเบียน {occ} ({100*occ/n:.3f}%)"
        f" · จุดทะเบียนที่ไม่มีเซลล์ของตัวเองในตาราง {len(outside_names)}")

LAD = pd.DataFrame(rows)
guard_variance(LAD.edge_km.values, "hex-edge-km", min_spread_km=1.0)
assert (LAD.cells.values[:-1] < LAD.cells.values[1:]).all(), "จำนวนเซลล์ต้องมากขึ้นทุกขั้น"
assert (LAD.edge_km.values[:-1] > LAD.edge_km.values[1:]).all(), "ขอบเซลล์ต้องสั้นลงทุกขั้น"

## ตอน 4b) ทะเบียนเองละเอียดได้แค่ไหน: จุดสองจุดอยู่ใกล้กันเท่าไร

In [ ]:
# --------------- 4b. ทะเบียนเองละเอียดได้แค่ไหน: จุดสองจุดอยู่ใกล้กันเท่าไร
DS = hav_km(SITES, SITES)
np.fill_diagonal(DS, np.inf)
i, j = np.unravel_index(DS.argmin(), DS.shape)
iu = np.triu_indices(len(df), 1)                 # เฉพาะคู่ที่ไม่ซ้ำ (เหนือเส้นทแยง)
PAIRS = dict(closest_pair_km=round(float(DS.min()), 3),
             closest_pair=[str(NAMES[i]), str(NAMES[j])],
             total_pairs=int(len(iu[0])),
             pairs_under_0p5km=int((DS[iu] < 0.5).sum()),
             pairs_under_1km=int((DS[iu] < 1.0).sum()),
             pairs_under_5km=int((DS[iu] < 5.0).sum()),
             pairs_under_20km=int((DS[iu] < 20.0).sum()))
guard_distances(np.array([[PAIRS["closest_pair_km"]]]), "site-pair-min", max_km=2500)
assert PAIRS["total_pairs"] == len(df) * (len(df) - 1) // 2, "นับคู่จุดต้องไม่ซ้ำไปมา"
assert PAIRS["pairs_under_1km"] <= PAIRS["total_pairs"], "จำนวนคู่มากกว่าคู่ที่เป็นไปได้ = นับซ้ำ"

# ---------- 4b2. แยก 83 จุดให้คนละเซลล์ต้องใช้ตารางละเอียดแค่ไหน (ไม่ต้องเติมทั้งประเทศ)
SEP = {}
for r in range(5, 13):
    SEP[r] = len({h3.latlng_to_cell(float(la), float(lo), r)
                  for la, lo in zip(df.latitude.values, df.longitude.values)})
PAIRS["distinct_site_cells_by_res"] = SEP
PAIRS["first_res_all_separate"] = next((r for r in range(5, 13) if SEP[r] == 83), None)
# ---------- 4b3. จุดทะเบียนที่เส้นขอบประเทศมาตรฐาน "มองไม่เห็น"
inside_pt = np.array([TH.contains(Point(lo, la))
                      for la, lo in zip(df.latitude.values, df.longitude.values)])
outside_pt = sorted(df.name.values[~inside_pt].astype(str))
PAIRS["sites_outside_country_outline"] = outside_pt
PAIRS["n_sites_outside_country_outline"] = len(outside_pt)

log(f"คู่จุดทะเบียนที่ใกล้ที่สุด {PAIRS['closest_pair_km']:,.3f} กม. "
    f"({PAIRS['closest_pair'][0]} ↔ {PAIRS['closest_pair'][1]}) · "
    f"คู่ต่ำกว่า 0.5 กม. {PAIRS['pairs_under_0p5km']} · ต่ำกว่า 1 กม. {PAIRS['pairs_under_1km']} · "
    f"ต่ำกว่า 5 กม. {PAIRS['pairs_under_5km']} · ต่ำกว่า 20 กม. {PAIRS['pairs_under_20km']} "
    f"(จากทั้งหมด {PAIRS['total_pairs']:,} คู่)")
log("จำนวนเซลล์ที่จุด 83 จุดครอง: " + " · ".join(f"res {k}={v}" for k, v in SEP.items()) +
    f" · แยกครบทุกคนที่ res {PAIRS['first_res_all_separate']}")
log(f"จุดทะเบียนที่อยู่นอกเส้นขอบประเทศของ Natural Earth 10m = {len(outside_pt)} จุด: "
    + ", ".join(outside_pt))

## ตอน 4c) จุดทะเบียนที่ "ไม่มีเซลล์ของตัวเอง" ในตารางของซีรีส์

In [ ]:
# ------------------- 4c. จุดทะเบียนที่ "ไม่มีเซลล์ของตัวเอง" ในตารางของซีรีส์
sc5 = [h3.latlng_to_cell(float(la), float(lo), 5)
       for la, lo in zip(df.latitude.values, df.longitude.values)]
miss_ne = sorted({str(NAMES[k]) for k, c in enumerate(sc5) if c not in mine5})
miss_d3 = sorted({str(NAMES[k]) for k, c in enumerate(sc5) if c not in day3})
NOSITE = dict(res5_ne10m_missing=miss_ne, res5_day3_missing=miss_d3,
              n_ne10m=len(miss_ne), n_day3=len(miss_d3),
              why="H3 เติมตารางแบบเก็บเซลล์ที่จุดกลางอยู่ในประเทศเท่านั้น "
                  "เซลล์ริมขอบที่กินเข้ามาในไทยแต่จุดกลางอยู่ต่างประเทศจึงไม่ถูกนับ "
                  "จุดทะเบียนที่อยู่กับเซลล์แบบนั้นจึงไม่มีที่อยู่บนกริด")
log(f"res 5: {len(miss_ne)} จุดที่ไม่มีเซลล์ของตัวเองในตาราง ({', '.join(miss_ne)}) · "
    f"เทียบกับกริด Day 3 = {len(miss_d3)} จุด")

## ตอน 5) ข้อมูลที่ได้เพิ่มต่อขั้น: ลูกตอบต่างจากแม่หรือไม่

In [ ]:
# ------------------------------------- 5. ข้อมูลที่ได้เพิ่มต่อขั้น: ลูกตอบต่างจากแม่หรือไม่
INHERIT = []
for r in RES_MEASURE[1:]:
    ch, pa = CELLS[r], CELLS[r - 1]
    setr = set(pa.astype(str))
    par = [h3.cell_to_parent(x, r - 1) for x in ch]
    keep = np.fromiter((str(p) in setr for p in par), bool, len(par))
    pos = {str(v): i for i, v in enumerate(pa)}
    pidx = np.fromiter((pos[str(p)] for p, k in zip(par, keep) if k), int, int(keep.sum()))
    child_ans = np.asarray(ANS[r], dtype=object)[keep]
    par_ans = np.asarray(ANS[r - 1], dtype=object)[pidx]
    same = float(np.mean(child_ans == par_ans))
    INHERIT.append(dict(step=f"{r-1}→{r}", children_matched=int(keep.sum()),
                        same_as_parent_pct=round(100 * same, 3),
                        new_answer_cells=int(round((1 - same) * keep.sum()))))
    log(f"{r-1}→{r}: ลูก {int(keep.sum()):,} เซลล์ ตอบเหมือนแม่ {100*same:.3f}% "
        f"(คำตอบใหม่ {int(round((1-same)*keep.sum())):,} เซลล์)")

## ตอน 6) ภาพรวม: res 8 เทียบแม่ res 5 ทั้งประเทศ

In [ ]:
# ------------------------------- 6. ภาพรวม: res 8 เทียบแม่ res 5 ทั้งประเทศ
par8to5 = [h3.cell_to_parent(x, 5) for x in CELLS[8]]
pos5 = {str(v): i for i, v in enumerate(CELLS[5])}
kid = np.fromiter((pos5.get(str(p), -1) for p in par8to5), int, len(par8to5))
valid = kid >= 0
agree85 = float(np.mean(np.asarray(ANS[8], dtype=object)[valid] ==
                        np.asarray(ANS[5], dtype=object)[kid[valid]]))
extra_km = DMIN[8][valid] - DMIN[5][kid[valid]]
log(f"res 8 เทียบแม่ res 5: ตอบตรงกัน {100*agree85:.2f}% "
    f"({int(round((1-agree85)*valid.sum())):,} เซลล์ต่าง) · "
    f"ระยะที่เปลี่ยนมากที่สุด {extra_km.min():+.2f} ถึง {extra_km.max():+.2f} กม.")

## ตอน 7) เพดานรายละเอียดของต้นทาง (เส้นขอบ Natural Earth 10m)

In [ ]:
# --------------------------- 7. เพดานรายละเอียดของต้นทาง (เส้นขอบ Natural Earth 10m)
_tr = Transformer.from_crs("EPSG:4326", EQ, always_xy=True)
segs = []
for p in PARTS:
    xy = np.array(p.exterior.coords)
    xs, ys = _tr.transform(xy[:, 0], xy[:, 1])
    segs.append(np.hypot(np.diff(xs), np.diff(ys)))
SEG = np.concatenate(segs)
assert SEG.max() > 1000, "หน่วยช่วงห่างจุดไม่ใช่เมตรหรือเส้นขอบสั้นผิดปกติ"
FLOOR = dict(border_segments=int(SEG.size), seg_min_m=float(SEG.min()),
             seg_p05_m=float(np.percentile(SEG, 5)), seg_median_m=float(np.median(SEG)),
             seg_mean_m=float(SEG.mean()), seg_p95_m=float(np.percentile(SEG, 95)),
             seg_max_m=float(SEG.max()))
log(f"ช่วงห่างจุดบนเส้นขอบ (NE 10m): ต่ำสุด {FLOOR['seg_min_m']:,.2f} · "
    f"มัธยฐาน {FLOOR['seg_median_m']:,.1f} ·สูงสุด {FLOOR['seg_max_m']:,.0f} ม. "
    f"({FLOOR['border_segments']:,} ช่วง)")
CROSSOVER = int(max(r for r in RES_MEASURE if LAD.loc[LAD.res == r, "edge_km"].iloc[0] * 1000
                    > FLOOR["seg_median_m"]))
log(f"res ที่ขอบเซลล์ยังหนากว่าช่วงห่างจุดของเส้นขอบ: ถึง res {CROSSOVER} — "
    f"res {CROSSOVER+1} คือจุดที่ตารางละเอียดกว่าต้นทาง")

## ตอน 8) ขนาดไฟล์จริง + เวลาวาดจริง (วัด ไม่ใช่คาด)

In [ ]:
# ---------------------------- 8. ขนาดไฟล์จริง + เวลาวาดจริง (วัด ไม่ใช่คาด)
def hex_ring(c, nd=5):
    b = h3.cell_to_boundary(c)
    return [(round(float(p[1]), nd), round(float(p[0]), nd)) for p in b]

SIZES = []
for r in BYTE_RES:
    path = os.path.join(OUT, f"_tmp_res{r}.geojson")
    feats = [{"type": "Feature", "properties": {"cell": str(c)},
              "geometry": {"type": "Polygon",
                           "coordinates": [hex_ring(c, 7) + [hex_ring(c, 7)[0]]]}}
             for c in CELLS[r]]
    with open(path, "w", encoding="utf-8") as f:
        json.dump({"type": "FeatureCollection", "crs": {"type": "name", "properties":
                  {"name": "urn:ogc:def:crs:OGC:1.3:CRS84"}}, "features": feats}, f)
    b = os.path.getsize(path)
    SIZES.append(dict(res=r, cells=int(len(CELLS[r])), mb=round(b / 1e6, 3),
                      bytes_per_cell=round(b / len(CELLS[r]), 1)))
    os.remove(path)
    log(f"GeoJSON res {r}: {b/1e6:,.3f} MB · {SIZES[-1]['bytes_per_cell']:,.1f} ไบต์/เซลล์")
BCD = float(np.median([s["bytes_per_cell"] for s in SIZES]))

DUR = {}
tmp = os.path.join(OUT, "_tmp_timing.png")
for r in TIME_RES:
    t1 = time.perf_counter()
    rings = [hex_ring(c, 5) for c in CELLS[r]]
    t2 = time.perf_counter()
    fig, ax = plt.subplots(figsize=(2.2, 2.6), dpi=72)
    ax.add_collection(PolyCollection(rings, facecolor="#5b7fa6", edgecolor="none"))
    ax.set_xlim(97, 106); ax.set_ylim(5, 21); ax.axis("off")
    fig.savefig(tmp, transparent=True)
    plt.close(fig); os.remove(tmp)
    t3 = time.perf_counter()
    DUR[r] = dict(cells=int(len(rings)), build_s=round(t2 - t1, 2), draw_s=round(t3 - t2, 2),
                  total_s=round(t3 - t1, 2), cells_per_s=round(len(rings) / (t3 - t1), 1))
    del rings
    log(f"จับเวลาวาด res {r}: {DUR[r]['cells']:,} เซลล์ · สร้างขอบ {DUR[r]['build_s']} วิ · "
        f"วาด+เซฟ {DUR[r]['draw_s']} วิ = {DUR[r]['cells_per_s']:,.0f} เซลล์/วิ")
THR = float(DUR[8]["cells_per_s"])

## ตอน 9) ลิมิตของ Borges (เลขคณิตล้วน)

In [ ]:
# ------------------------------------------------ 9. ลิมิตของ Borges (เลขคณิตล้วน)
def cells_at_edge(edge_m):
    return (REF_KM2 * 1e6) / (HEX_K * edge_m ** 2)

hair_cells = cells_at_edge(HAIR_M)
dot_cells = cells_at_edge(DOT300_M)
LIMIT = dict(
    hair_edge_mm=HAIR_M * 1000, dot300_edge_mm=DOT300_M * 1000,
    hair_cells=float(hair_cells), dot300_cells=float(dot_cells),
    h3_max_res=15, h3_res15_edge_mm=float(h3.average_hexagon_edge_length(15, "km")) * 1e6,
    steps_beyond_res8_to_hair=int(np.ceil(np.log(hair_cells / len(CELLS[8])) / np.log(7))),
    hair_bytes_at_1B_per_cell=float(hair_cells), hair_bytes_geojson=float(hair_cells * BCD),
    hair_draw_years=float(hair_cells / THR / 3600 / 24 / 365.25),
    dot300_draw_years=float(dot_cells / THR / 3600 / 24 / 365.25),
    one_to_one_a4_sheets=float(REF_KM2 * 1e6 / A4_M2), one_to_one_paper_km2=REF_KM2,
    bytes_per_cell_measured=BCD, cells_per_second_measured=THR)
log(f"ผมหนึ่งเส้น: {hair_cells:,.0f} เซลล์ · ซอยต่อจาก res 8 อีก "
    f"{LIMIT['steps_beyond_res8_to_hair']} ขั้น · วาดที่ {THR:,.0f} เซลล์/วิ = "
    f"{LIMIT['hair_draw_years']:,.3g} ปี")

## ตอน 10) ภาพ 1 — บันไดความละเอียด

In [ ]:
# ============================================= 10. ภาพ 1 — บันไดความละเอียด
def draw_ladder():
    fig = plt.figure(figsize=(16.2, 7.4), dpi=150)
    fig.patch.set_facecolor(BG)
    gs = fig.add_gridspec(1, len(RES_DRAW), left=0.015, right=0.985, wspace=0.10,
                          hspace=0.1, top=0.86)
    shade = ["#22334c", "#2a4661", "#33627e", "#3d8299", "#57b0bf", "#e8c46a"]
    for i, r in enumerate(RES_DRAW):
        ax = fig.add_subplot(gs[0, i]); ax.set_facecolor(BG)
        rings = [hex_ring(c, 5) for c in CELLS[r]]
        ax.add_collection(PolyCollection(rings, facecolor=shade[i], edgecolor=BG,
                                         linewidth=0.25 if r <= 6 else 0.0))
        for p in PARTS:
            x, y = p.exterior.xy
            ax.plot(x, y, color=INK, lw=0.7, alpha=0.9, zorder=4)
        if r <= 6:
            sc = np.array([h3.latlng_to_cell(float(la), float(lo), r)
                           for la, lo in zip(df.latitude.values, df.longitude.values)], dtype=object)
            keep = np.isin(sc, CELLS[r].astype(str))
            ax.scatter(df.longitude.values[keep], df.latitude.values[keep], s=16, color="#0d1522",
                       edgecolor="#ffffff", linewidth=0.7, zorder=6)
        ax.set_xlim(97.2, 106.4); ax.set_ylim(4.8, 21.2); ax.set_aspect("equal"); ax.axis("off")
        w = LAD[LAD.res == r].iloc[0]
        ax.set_title(T(f"res {r} · {int(w.cells):,} เซลล์\n"
                       f"ขอบ {w.edge_km:,.2f} กม.\n"
                       f"จุดทะเบียนอยู่ใน {int(w.occupied_cells)} เซลล์ ({w.occupied_pct:.2f}%)"),
                     color=INK, fontsize=10, linespacing=1.5, pad=8)
        del rings
    fig.text(0.5, 0.012, T("จุดขาวคือแหล่งในทะเบียน 83 จุด · res 8 มีเซลล์ 597,542 ก้อน "
                           "จนทั้งประเทศกลายเป็นแผ่นสีเดียว — แผนที่เริ่มหายไป เหลือแต่ดินแดน"),
             color=MUTED, fontsize=11, ha="center")
    fig.suptitle(T("บันไดความละเอียด · ราคาเพิ่มเจ็ดเท่าทุกขั้น ข้อมูลชุดเดิม 83 จุด"),
                 color=INK, fontsize=16, y=0.99)
    p = os.path.join(OUT, "day14_ladder.png")
    fig.savefig(p, facecolor=BG, bbox_inches="tight"); plt.close(fig)
    log(f"เขียน {p}")
    return p

## ตอน 11) ภาพ 2 — เส้นราคากับเส้นข้อมูล

In [ ]:
# ============================================= 11. ภาพ 2 — เส้นราคากับเส้นข้อมูล
def draw_curves():
    fig, axes = plt.subplots(1, 2, figsize=(14.6, 5.9), dpi=150)
    fig.patch.set_facecolor(BG)
    x = LAD.res.values.astype(float)
    ax = axes[0]; ax.set_facecolor("#111c2b")
    ax.plot(x, LAD.cells.values, "-o", color=ACCENT, lw=2.2, ms=6, label=T("จำนวนเซลล์"))
    ax.plot(x, LAD.vertices.values, "--s", color=COOL, lw=1.6, ms=5,
            label=T("จุดที่ต้องวาด (เซลล์ × 6)"))
    ax.plot([s["res"] for s in SIZES], [s["mb"] * 1e6 for s in SIZES], "^", color=TEAL, ms=8, ls="none",
            label=T("ขนาดไฟล์ GeoJSON (ไบต์) ที่วัดจริง"))
    fit = np.polyfit(x[2:], np.log10(LAD.cells.values[2:]), 1)
    ax.plot(x, 10 ** np.polyval(fit, x), ":", color=ACCENT, alpha=0.4, lw=1.2,
            label=T("เส้นเจ็ดเท่าต่อขั้น (res 4–8)"))
    ax.set_yscale("log"); ax.set_xlabel(T("ความละเอียด H3 (res)"))
    ax.set_ylabel(T("ราคาของแผนที่ (มาตราลอการิทึม)"))
    ax.set_title(T("เส้นที่ 1 — ราคา: โตขึ้นเจ็ดเท่าทุกขั้น ไม่อิ่มตัว"), color=INK, fontsize=12.5)
    ax.tick_params(colors=INK, labelsize=10); ax.grid(color="#26364f", lw=0.6, alpha=0.6)
    ax.legend(facecolor="#111c2b", edgecolor="#33415e", labelcolor=INK, fontsize=9)
    for s in ax.spines.values(): s.set_color("#33415e")
    for s in ("top", "right"): ax.spines[s].set_visible(False)

    ax = axes[1]; ax.set_facecolor("#111c2b")
    ax.plot(x, LAD.occupied_pct.values, "-o", color=TEAL, lw=2.2, ms=6,
            label=T("ร้อยละของเซลล์ที่มีจุดทะเบียน"))
    ax.plot(x, np.abs(LAD.nominal_vs_country_pct.values - 100), "-s", color=WARM, lw=2.2, ms=6,
            label=T("ค่าสัมบูรณ์ของผลต่างพื้นที่ตารางกับพื้นที่ประเทศ (ร้อยละ)"))
    ax.plot([int(i["step"].split("→")[1]) for i in INHERIT],
            [100 - i["same_as_parent_pct"] for i in INHERIT], "-D", color=COOL, lw=2.0, ms=6,
            label=T("ร้อยละของเซลล์ลูกที่ตอบต่างจากแม่ (ข้อมูลใหม่ที่ซื้อได้)"))
    ax.set_yscale("log"); ax.set_xlabel(T("ความละเอียด H3 (res)"))
    ax.set_ylabel(T("ร้อยละ (มาตราลอการิทึม)"))
    ax.set_title(T("เส้นที่ 2 — ข้อมูล: ตกแรงแล้วแบนราบ อิ่มตัวตั้งแต่ res 5–6"),
                 color=INK, fontsize=12.5)
    ax.tick_params(colors=INK, labelsize=10); ax.grid(color="#26364f", lw=0.6, alpha=0.6)
    ax.legend(facecolor="#111c2b", edgecolor="#33415e", labelcolor=INK, fontsize=9)
    for s in ax.spines.values(): s.set_color("#33415e")
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    fig.suptitle(T("สองเส้นนี้คือเนื้อความของแผนที่ที่ “ละเอียดจนกลายเป็นดินแดน”: "
                   "ฝั่งซ้ายจ่ายไม่หยุด ฝั่งขวาอิ่มตัวตั้งแต่ res 5"), color=INK, fontsize=14.5, y=0.995)
    p = os.path.join(OUT, "day14_detail_vs_info.png")
    fig.savefig(p, facecolor=BG, bbox_inches="tight"); plt.close(fig)
    log(f"เขียน {p}")
    return p

## ตอน 12) ภาพ 3 — ไม้บรรทัดความละเอียด + เลขคณิตของ Borges

In [ ]:
# ============================================= 12. ภาพ 3 — ไม้บรรทัดความละเอียด + เลขคณิตของ Borges
def draw_floor():
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(15.6, 5.8), dpi=150,
                                  gridspec_kw=dict(width_ratios=[1.32, 1.0]))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor("#111c2b")
    marks = [(float(LAD.loc[LAD.res == r, "edge_km"].iloc[0]) * 1e6,
              T(f"res {r} · {int(LAD.loc[LAD.res == r, 'cells'].iloc[0]):,} เซลล์"), "grid")
             for r in RES_MEASURE]
    marks += [(FLOOR["seg_median_m"],
               T(f"ช่วงห่างจุดบนเส้นขอบประเทศ (NE 10m) · มัธยฐาน {FLOOR['seg_median_m']:,.0f} ม."), "floor"),
              (LIMIT["h3_res15_edge_mm"],
               T(f"ขอบเซลล์ที่ละเอียดที่สุดที่ H3 ทำได้ · res {LIMIT['h3_max_res']} · "
                 f"{LIMIT['h3_res15_edge_mm']:,.1f} มม."), "floor"),
              (DOT300_M * 1000, T(f"จุดหมึกหนึ่งจุดที่ 300 dpi · {DOT300_M*1000:.3f} มม."), "hair"),
              (HAIR_M * 1000, T("ผมหนึ่งเส้น 0.05 มม. · ขีดสุดที่ Borges เรียก"), "hair")]
    col = {"grid": ACCENT, "floor": COOL, "hair": WARM}
    logmid = np.sqrt(min(v for v, _, _ in marks) * max(v for v, _, _ in marks))
    for i, (v, lab, kind) in enumerate(marks):
        y = len(marks) - i
        ax.scatter([v], [y], s=70, color=col[kind], zorder=5)
        if v > logmid:
            ax.annotate(lab, (v, y), xytext=(-12, 0), textcoords="offset points",
                        color=INK, fontsize=10, va="center", ha="right")
        else:
            ax.annotate(lab, (v, y), xytext=(12, 0), textcoords="offset points",
                        color=INK, fontsize=10, va="center")
    ax.axvline(FLOOR["seg_median_m"], color=COOL, lw=1.2, ls="--", alpha=0.75)
    lo = min(v for v, _, _ in marks) * 0.55
    ax.axvspan(lo, FLOOR["seg_median_m"], color=WARM, alpha=0.06)
    ax.set_xscale("log"); ax.set_xlim(lo, max(v for v, _, _ in marks) * 2.2)
    ax.set_ylim(0.5, len(marks) + 0.6); ax.set_yticks([])
    ax.set_xlabel(T("ความยาวขอบเซลล์ (มิลลิเมตร · มาตราลอการิทึม)"), labelpad=16)
    ax.set_title(T("ไม้บรรทัดความละเอียด: ตรงไหนละเอียดจริง ตรงไหนละเอียดเพราะแต่ง"),
                 color=INK, fontsize=12.5)
    ax.tick_params(colors=INK, labelsize=10)
    for s in ("top", "right", "left"): ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color("#33415e")

    ax2.set_facecolor("#111c2b"); ax2.axis("off")
    lines = [
        (T("ถ้าแผนที่ละเอียดเท่าผมหนึ่งเส้น ที่มาตราส่วน 1 ต่อ 1"), ""),
        (T("จำนวนหกเหลี่ยมที่ต้องมี"), f"{LIMIT['hair_cells']:,.0f} " + T("เซลล์")),
        (T("ต้องซอยต่อจาก res 8 อีก"), f"{LIMIT['steps_beyond_res8_to_hair']} " + T("ขั้น (ขั้นละเจ็ดเท่า)")),
        (T("ไฟล์ถ้าเก็บหนึ่งไบต์ต่อเซลล์"), f"{LIMIT['hair_bytes_at_1B_per_cell']/1e18:,.1f} " + T("เอ็กซาไบต์")),
        (T("ไฟล์แบบ GeoJSON (ไบต์ต่อเซลล์วัดจริง)"),
         f"{LIMIT['hair_bytes_geojson']/1e21:,.1f} " + T("เซตตาไบต์")),
        (T(f"เวลาวาดที่ {THR:,.0f} เซลล์ต่อวินาที (วัดจริงที่ res 8)"),
         f"{LIMIT['hair_draw_years']:,.0f} " + T("ปี")),
        (T("พิมพ์ 1:1 ด้วยกระดาษ A4"), f"{LIMIT['one_to_one_a4_sheets']:,.0f} " + T("แผ่น")),
        (T("เท่ากับเนื้อที่กระดาษ"), f"{LIMIT['one_to_one_paper_km2']:,.0f} " + T("ตารางกิโลเมตร")),
        ("", ""),
        (T("อีกทางหนึ่ง ดินแดนละเอียดกว่าแผนที่"), ""),
        (T("จุดในทะเบียนที่เส้นขอบประเทศมาตรฐานมองไม่เห็น"),
         f"{PAIRS['n_sites_outside_country_outline']} " + T("จาก 83 จุด")),
        (T("res ที่ต้องใช้อยากให้ 83 จุดอยู่คนละเซลล์"), f"res {PAIRS['first_res_all_separate']}"),
    ]
    y = 0.93
    for k, v in lines:
        if not k:
            y -= 0.045; continue
        ax2.text(0.03, y, k, color=MUTED if not v else INK, fontsize=11, va="top")
        if v:
            ax2.text(0.97, y, v, color=ACCENT, fontsize=11, va="top", ha="right")
        y -= 0.083 if v else 0.048
    ax2.set_title(T("เลขคณิตของ Borges (ต่อจากกฎเจ็ดเท่า ไม่ใช่แผนที่ที่วาดจริง)"),
                  color=INK, fontsize=12)
    fig.text(0.26, 0.005, T("แถบสีส้มทางซ้ายของเส้นประ = ช่วงที่ต้นทางข้อมูลไม่มีรายละเอียดให้ลอกแล้ว"),
             color=WARM, fontsize=10, ha="center")
    fig.suptitle(T("แผนที่ละเอียดเท่าดินแดนเป็นไปไม่ได้ทั้งสองทาง: "
                   "ทางแรกต้นทางหมดรายละเอียดก่อน ทางที่สองราคาไม่มีวันหมด"),
                 color=INK, fontsize=15, y=0.99)
    p = os.path.join(OUT, "day14_floor.png")
    fig.savefig(p, facecolor=BG, bbox_inches="tight"); plt.close(fig)
    log(f"เขียน {p}")
    return p

P1, P2, P3 = draw_ladder(), draw_curves(), draw_floor()

## ตอน 13) เขียน artifact

In [ ]:
# ============================================= 13. เขียน artifact
def pyo(o):
    if hasattr(o, "item"):
        return o.item()
    if isinstance(o, (np.ndarray,)):
        return o.tolist()
    raise TypeError(f"เข้ารหัส JSON ไม่ได้ที่ {type(o)}")

LAD.round(6).to_csv(os.path.join(OUT, "day14_ladder.csv"), index=False)
STATS = dict(
    theme="A map so detailed it becomes the territory.",
    theme_source="30daymapchallenge.com หัวข้อ Day 14 “Borgesian map” (อ่านสดวันเขียน) · "
                 "อ้างถึง On Exactitude in Science ของ Jorge Luis Borges พิมพ์ครั้งแรกปี 1946",
    generated_local=time.strftime("%Y-%m-%d %H:%M:%S"),
    h3_version=h3.__version__,
    reference_areas=dict(natural_earth_thailand_km2_used=REF_KM2,
                         recomputed_this_run_km2=round(country_m, 1)),
    ladder=[{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()} for r in rows],
    ratio_seven_law={f"{r}→{r+1}": round(v, 3) for r, v in RATIO.items()},
    inheritance=INHERIT,
    res8_vs_res5=dict(agree_pct=round(100 * agree85, 3),
                      different_cells=int(round((1 - agree85) * valid.sum())),
                      distance_change_min_km=round(float(extra_km.min()), 3),
                      distance_change_max_km=round(float(extra_km.max()), 3)),
    geojson_bytes_measured=SIZES,
    render_seconds_measured={str(k): v for k, v in DUR.items()},
    border_detail_floor=FLOOR,
    crossover=dict(last_res_coarser_than_border_source=CROSSOVER,
                   border_median_spacing_m=round(FLOOR["seg_median_m"], 1)),
    borgesian_limit={k: (round(v, 4) if isinstance(v, float) else v) for k, v in LIMIT.items()},
    reconciliation_day3=REC,
    site_register_total=int(len(df)),
    register_detail=PAIRS,
    sites_without_own_cell=NOSITE,
    extremes=dict(finest_edge_mm=round(float(LAD.edge_km.min()) * 1e6, 2),
                  coarsest_edge_mm=round(float(LAD.edge_km.max()) * 1e6, 1),
                  cells_min=int(LAD.cells.min()), cells_max=int(LAD.cells.max()),
                  occupied_pct_min=round(float(LAD.occupied_pct.min()), 4),
                  occupied_pct_max=round(float(LAD.occupied_pct.max()), 3)),
    figures=["day14_ladder.png", "day14_detail_vs_info.png", "day14_floor.png"],
    metric_definitions={
        "res": "ระดับตาราง H3 (แต่ละขั้นแบ่งแม่หนึ่งเซลล์เป็นลูกเจ็ดเซลล์ พื้นที่เล็กลงเจ็ดเท่า)",
        "cells": "จำนวนเซลล์ H3 ที่จุดกลางเซลล์อยู่ในเส้นขอบประเทศ Natural Earth 10m (นับใหม่รอบนี้)",
        "edge_km": "ความยาวขอบเซลล์เฉลี่ยที่ระดับนั้น ตามค่าที่ H3 รายงาน (ค่าเฉลี่ยทั้งโลก หน่วยกิโลเมตร)",
        "cell_area_km2": "พื้นที่เซลล์เต็มเฉลี่ย (ตารางกิโลเมตร) สุ่มวัด 20,000 เซลล์ต่อขั้นด้วย h3.cell_area (seed 14)",
        "nominal_area_km2": "cell_area_km2 × cells คือพื้นที่ถ้าถือว่าทุกเซลล์เป็นก้อนเต็ม (ไม่ตัดตามขอบ)",
        "nominal_vs_country_pct": "nominal_area_km2 ÷ 512,593.1 × 100 (มากกว่า 100 คือตารางล้นประเทศ น้อยกว่า 100 คือตารางไม่คลุม)",
        "occupied_cells": "จำนวนเซลล์ที่มีแหล่งในทะเบียน 83 จุด อย่างน้อยหนึ่งจุดตกอยู่ (นับเฉพาะเซลล์ที่อยู่ในชุดของประเทศ)",
        "site_cells_distinct": "จำนวนเซลล์ที่แตกต่างกันที่เกิดขึ้นจริงจาก 83 จุด (ไม่สนใจว่าเซลล์นั้นอยู่ในชุดของประเทศไหม)",
        "distinct_site_cells_by_res": "จำนวนเซลล์ที่จุด 83 จุดครอง ที่ res 5 ถึง 12 — บอกว่าต้องละเอียดแค่ไหนกว่าทุกจุดจะแยกจากกัน",
        "sites_outside_country_outline": "จุดในทะเบียนที่พิกัดของมันอยู่นอกเส้นขอบประเทศของ Natural Earth 10m (ต้นทางมองไม่เห็น)",
        "site_cells_outside_fill": "เซลล์ที่มีจุดทะเบียนตกอยู่แต่จุดกลางของเซลล์อยู่นอกชุดที่เติมจากประเทศ (เกิดริมขอบ)",
        "sites_without_own_cell": "รายชื่อจุดทะเบียนที่เซลล์ของตัวไม่อยู่ในตาราง (res 5) ทั้งของวันนี้และของ Day 3 เพราะ H3 เก็บเซลล์แบบจุดกลางอยู่ในประเทศ",
        "register_detail": "ระยะระหว่างจุดสองจุดในทะเบียน (กิโลเมตร) นับเฉพาะคู่ไม่ซ้ำ บอกว่าตัวข้อมูลเองละเอียดได้แค่เท่านั้น "
                           "คู่ที่ใกล้ที่สุดคือสองแหล่งที่ลงทะเบียนคนละแห่งแต่อยู่ติดกัน",
        "first_res_all_separate": "res แรกที่ทั้ง 83 จุดอยู่คนละเซลล์ หากเป็น null แปลว่าแม้แต่ res 8 ยังแยกไม่ครบ",
        "occupied_pct": "occupied_cells ÷ cells × 100 คือสัดส่วนของแผนที่ที่มีอะไรอยู่ ไม่ใช่ช่องว่าง",
        "same_as_parent_pct": "ร้อยละของเซลล์ลูกที่ตอบ “จุดทะเบียนที่ใกล้ที่สุด” เหมือนจุดกลางของแม่ — รายละเอียดเพิ่มแต่คำตอบไม่เปลี่ยน",
        "new_answer_cells": "จำนวนเซลล์ลูกที่ตอบต่างจากแม่ คือเนื้อหาใหม่ที่การซอยครั้งนั้นซื้อมาได้",
        "seg_median_m": "ค่ามัธยฐานของระยะระหว่างจุดต่อเนื่องบนเส้นขอบประเทศในไฟล์ Natural Earth 10m (เมตร) — เพดานรายละเอียดของต้นทาง",
        "crossover": "res สุดท้ายที่ขอบเซลล์ยังยาวกว่าช่วงห่างจุดบนเส้นขอบ เลยจุดนี้ตารางละเอียดกว่าเส้นที่ลอกมา",
        "hair_cells": "จำนวนหกเหลี่ยมขนาด 0.05 มม. ที่ต้องวางทับพื้นที่ 512,593.1 ตารางกิโลเมตร คำนวณจากสูตรพื้นที่หกเหลี่ยม 2.598·s²",
        "cells_per_second_measured": "จำนวนเซลล์ต่อวินาทีที่ชุดโค้ดนี้วาดแล้วเซฟเป็นไฟล์ได้จริงที่ res 8 (จับเวลา)",
        "hair_draw_years": "hair_cells ÷ cells_per_second_measured แปลงเป็นปี สมมติให้เครื่องทำงานต่อเนื่องไม่หยุด",
        "one_to_one_a4_sheets": "พื้นที่ประเทศ ÷ พื้นที่แผ่น A4 (0.210 × 0.297 เมตร ตาม ISO 216) คือจำนวนแผ่นถ้าพิมพ์ที่มาตราส่วน 1 ต่อ 1"},
    note="res 9 ขึ้นไปไม่ได้วาดจริง (4,182,974 เซลล์ ใช้เวลาเติมราวหนึ่งนาที) ตัวเลขส่วนหลังต่อจากกฎเจ็ดเท่า และบอกว่าต่อจากกฎ")
json.dump(STATS, open(os.path.join(OUT, "day14_stats.json"), "w", encoding="utf-8"),
          ensure_ascii=False, indent=2, default=pyo)
log(f"เขียน day14_stats.json และ day14_ladder.csv ที่ {OUT}")

## ตอน 14) referee

In [ ]:
# ============================================= 14. referee
for f in ["day14_ladder.png", "day14_detail_vs_info.png", "day14_floor.png",
          "day14_ladder.csv", "day14_stats.json"]:
    pth = os.path.join(OUT, f)
    assert os.path.exists(pth) and os.path.getsize(pth) > 0, f"ไฟล์ขาดหรือว่าง: {f}"
assert not [f for f in os.listdir(OUT) if f.startswith("_tmp")], "ต้องลบไฟล์ชั่วคราวให้หมด"
json.load(open(os.path.join(OUT, "day14_stats.json"), encoding="utf-8"))
lint_thai(" ".join(TXT) + " " + __doc__, "day14-figure-texts")
# สแกน unicode ทั้งไฟล์: ห้ามมีอักษรเวียดนาม/จีน/เกาหลีหลุดมาในบรรทัดที่มีภาษาไทย
_src_path = os.path.abspath(__file__) if "__file__" in globals() else None
_bad = []
if _src_path and os.path.exists(_src_path):
    _lines = open(_src_path, encoding="utf-8").read().split(chr(10))
    for _n, _l in enumerate(_lines, 1):
        if any(0x0E00 <= ord(ch) <= 0x0E7F for ch in _l):
            for ch in _l:
                o = ord(ch)
                if (0x0100 <= o <= 0x017F) or (0x1E00 <= o <= 0x1EFF) or \
                   (0x3000 <= o <= 0x30FF) or (0x3400 <= o <= 0x4DBF) or \
                   (0x4E00 <= o <= 0x9FFF) or (0xAC00 <= o <= 0xD7AF):
                    _bad.append((_n, ch))
    assert not _bad, f"พบอักษรไม่ใช่ไทย/ASCII ในบรรทัดที่มีภาษาไทย: {_bad[:6]}"
    log("ตรวจ unicode ทั้งไฟล์แล้ว: ไม่มีอักษรเวียดนาม/จีน/เกาหลีหลุดมา")
else:
    log("รันใน Colab ไม่มีการ์ดไฟล์ต้นทาง จึงข้ามการสแกน unicode")
a5 = float(LAD.loc[LAD.res == 5, "nominal_area_km2"].iloc[0])
log(f"res 5 nominal {a5:,.1f} ตร.กม. (Day 12 บันทึกพื้นที่ที่ตัดขอบแล้ว 496,867.2 · ต่าง "
    f"{100*(a5-496867.2)/496867.2:+.2f}% เพราะวันนี้ไม่นับการตัดขอบ)")
summary = dict(
    res2_cells=int(LAD.loc[LAD.res == 2, "cells"].iloc[0]),
    res5_cells=int(LAD.loc[LAD.res == 5, "cells"].iloc[0]),
    res8_cells=int(LAD.loc[LAD.res == 8, "cells"].iloc[0]),
    res5_occupied_pct=round(float(LAD.loc[LAD.res == 5, "occupied_pct"].iloc[0]), 3),
    res8_occupied_pct=round(float(LAD.loc[LAD.res == 8, "occupied_pct"].iloc[0]), 4),
    first_res_all_separate=PAIRS["first_res_all_separate"],
    same_parent_7to8_pct=INHERIT[-1]["same_as_parent_pct"],
    agree_res8_res5_pct=round(100 * agree85, 2),
    border_median_m=round(FLOOR["seg_median_m"], 1),
    crossover_res=CROSSOVER,
    hair_cells=f"{LIMIT['hair_cells']:,.0f}",
    hair_draw_years=round(LIMIT["hair_draw_years"]),
    a4_one_to_one=f"{LIMIT['one_to_one_a4_sheets']:,.0f}")
log("สรุป: " + json.dumps(summary, ensure_ascii=False))
print("DONE day14 ใน %.1f วิ" % (time.time() - T0))
_LOGF.close()